In [0]:
from pyspark.sql.types import *

uber_schema = StructType([
    StructField("start_date", TimestampType(), True),
    StructField("end_date", TimestampType(), True),
    StructField("category", StringType(), True),
    StructField("start", StringType(), True),
    StructField("stop", StringType(), True),
    StructField("miles", FloatType(), True),
    StructField("purpose", StringType(), True)
])
from datetime import datetime

uber_data = [
    (datetime(2016, 1, 1, 21, 11), datetime(2016, 1, 1, 21, 17), "Business", "Fort Pierce", "Fort Pierce", 5.1, "Meal/Entertain"),
    (datetime(2016, 1, 2, 1, 25),  datetime(2016, 1, 2, 1, 37), "Business", "Fort Pierce", "Fort Pierce", 5.0, None),
    (datetime(2016, 1, 2, 20, 25), datetime(2016, 1, 2, 20, 38), "Business", "Fort Pierce", "Fort Pierce", 4.8, "Errand/Supplies"),
    (datetime(2016, 1, 5, 17, 31), datetime(2016, 1, 5, 17, 45), "Business", "Fort Pierce", "Fort Pierce", 4.7, "Meeting"),
    (datetime(2016, 1, 6, 14, 42), datetime(2016, 1, 6, 15, 49), "Business", "Fort Pierce", "West Palm Beach", 63.7, "Customer Visit"),
    (datetime(2016, 1, 6, 17, 15), datetime(2016, 1, 6, 17, 19), "Business", "West Palm Beach", "West Palm Beach", 4.3, "Meal/Entertain"),
    (datetime(2016, 1, 6, 17, 30), datetime(2016, 1, 6, 17, 35), "Business", "West Palm Beach", "Palm Beach", 7.1, "Meeting")
]

uber_df = spark.createDataFrame(uber_data, uber_schema)
uber_df.show(truncate=False)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col,row_number,desc,sum,round

uber_df2=uber_df.withColumn("sum_miles",round(sum("miles").over(Window.partitionBy("purpose").orderBy(desc("miles"))),2))


uber_df2.withColumn("rank_id",row_number().over(Window.orderBy(desc("sum_miles")))).filter(col("rank_id")<=3).show()

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import to_timestamp

worker_data = [
    (1, "John", "Doe", 80000, "2020-01-15", "Engineering"),
    (2, "Jane", "Smith", 120000, "2019-03-10", "Marketing"),
    (3, "Alice", "Brown", 120000, "2021-06-21", "Sales"),
    (4, "Bob", "Davis", 75000, "2018-04-30", "Engineering"),
    (5, "Charlie", "Miller", 95000, "2021-01-15", "Sales")
]

worker_columns = [
    "worker_id",
    "first_name",
    "last_name",
    "salary",
    "joining_date",
    "department"
]

worker_df = spark.createDataFrame(worker_data, worker_columns) \
    .withColumn("joining_date", to_timestamp("joining_date"))

worker_df.show()
worker_df.printSchema()

title_data = [
    (1, "Engineer", "2020-01-15"),
    (2, "Marketing Manager", "2019-03-10"),
    (3, "Sales Manager", "2021-06-21"),
    (4, "Junior Engineer", "2018-04-30"),
    (5, "Senior Salesperson", "2021-01-15")
]

title_columns = [
    "worker_ref_id",
    "worker_title",
    "affected_from"
]

title_df = spark.createDataFrame(title_data, title_columns) \
    .withColumn("affected_from", to_timestamp("affected_from"))

title_df.show()
title_df.printSchema()

joined_df = worker_df.join(
    title_df,
    worker_df.worker_id == title_df.worker_ref_id,
    "inner"
)

joined_df.show()



In [0]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window
window_spec = Window.orderBy(desc(col("salary")))
joined_df2 = joined_df.withColumn(
    "rank_id",
    dense_rank().over(window_spec)
)
display(joined_df2)
joined_df2.select("department").filter(col("rank_id")==1).show()